# ICA Pipeline

Пайплайн из трёх шагов:
1. **Шаг 1** — подобрать ICA для всех субъектов и сохранить `-ica.fif`
2. **Шаг 2** — визуализировать компоненты и заполнить `BAD_COMPONENTS`
3. **Шаг 3** — применить ICA, нарезать эпохи вокруг меток, сохранить

In [ ]:
import matplotlib
import matplotlib.axes

# Фикс совместимости MNE + matplotlib >= 3.8
if not hasattr(matplotlib.axes.Axes, '_axes_class'):
    matplotlib.axes.Axes._axes_class = matplotlib.axes.Axes

import mne
import numpy as np
from pathlib import Path

mne.set_log_level('WARNING')

BASE = Path(r'D:\Inner Speech Dataset\Inner Speech Dataset')
EMG_CHANNELS = ['B1+', 'B1-', 'B2+', 'B2-', 'EMG1', 'EMG2', 'Pharing1', 'Pharing2']

# ── Настройки эпох ────────────────────────────────────────────
TMIN     = -0.5   # сек до метки
TMAX     =  2.0   # сек после метки
BASELINE = (None, 0)

# ── Настройки ICA ─────────────────────────────────────────────
N_COMPONENTS = 20
RANDOM_STATE = 42
ICA_HP       = 1.0
ICA_LP       = 100.0

LANGUAGES = ['Russian', 'Spanish']

---
## Шаг 1 — Подобрать ICA и сохранить `-ica.fif`

Запускается один раз. Сохраняет ICA-объект рядом с данными субъекта.

In [ ]:
# Диагностика: каналы и ранг данных по каждому субъекту
for language in LANGUAGES:
    for edf in sorted((BASE / language).rglob('*.edf')):
        subject = edf.parent.name
        raw = mne.io.read_raw_edf(edf, preload=True, verbose=False)
        drop = [ch for ch in EMG_CHANNELS if ch in raw.ch_names]
        raw.drop_channels(drop)
        rank = mne.compute_rank(raw, rank='info')['eeg']
        print(f'[{language}] {subject}: {len(raw.ch_names)} EEG каналов, ранг={rank}')

In [ ]:
def fit_ica_all(languages=LANGUAGES, overwrite=False):
    for language in languages:
        edf_files = sorted((BASE / language).rglob('*.edf'))
        for edf in edf_files:
            subject  = edf.parent.name
            ica_path = edf.parent / f'{subject}-ica.fif'

            if ica_path.exists() and not overwrite:
                print(f'[{language}] {subject} — уже есть, пропускаем')
                continue

            print(f'[{language}] {subject} — подбираем ICA...', end=' ')
            raw = mne.io.read_raw_edf(edf, preload=True, verbose=False)

            drop = [ch for ch in EMG_CHANNELS if ch in raw.ch_names]
            raw.drop_channels(drop)

            montage = mne.channels.make_standard_montage('standard_1020')
            raw.set_montage(montage, match_case=False, on_missing='ignore')

            raw_filt = raw.copy().filter(ICA_HP, ICA_LP, verbose=False)

            ica = mne.preprocessing.ICA(
                n_components=N_COMPONENTS,
                random_state=RANDOM_STATE,
                max_iter='auto',
            )
            ica.fit(raw_filt)
            ica.save(ica_path, overwrite=True)
            print(f'сохранено -> {ica_path.name} ({ica.n_components_} компонент)')


# Пересчитать только испанский датасет:
fit_ica_all(languages=['Spanish'], overwrite=True)

# Пересчитать всё:
# fit_ica_all(languages=LANGUAGES, overwrite=True)

---
## Шаг 2 — Визуализация компонент

Запускай поблочно для каждого субъекта.
После осмотра заполни `BAD_COMPONENTS` в следующей ячейке.

In [ ]:
%matplotlib qt

# Выбери субъекта для осмотра
INSPECT_LANGUAGE = 'Russian'
INSPECT_SUBJECT  = 'sub1'

edf_path = BASE / INSPECT_LANGUAGE / INSPECT_SUBJECT / f'{INSPECT_SUBJECT}.edf'
ica_path = edf_path.parent / f'{INSPECT_SUBJECT}-ica.fif'

raw_inspect = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
drop = [ch for ch in EMG_CHANNELS if ch in raw_inspect.ch_names]
raw_inspect.drop_channels(drop)
montage = mne.channels.make_standard_montage('standard_1020')
raw_inspect.set_montage(montage, match_case=False, on_missing='ignore')
raw_filt_inspect = raw_inspect.copy().filter(ICA_HP, ICA_LP, verbose=False)

ica_inspect = mne.preprocessing.read_ica(ica_path)
n_comp = ica_inspect.n_components_
print(f'Компонент в ICA: {n_comp}')

ica_inspect.plot_components(inst=raw_filt_inspect, picks=range(n_comp))

In [ ]:
%matplotlib qt
ica_inspect.plot_sources(raw_filt_inspect, block=True)

In [ ]:
# Детальный осмотр конкретной компоненты
# ica_inspect.plot_properties(raw_filt_inspect, picks=[0, 1])

---
## Шаг 3 — Задать плохие компоненты и пересобрать fif

Заполни словарь `BAD_COMPONENTS` для каждого субъекта.
Формат: `{'Russian/sub1': [0, 2], 'Spanish/sub0': [1], ...}`

In [ ]:
# ── Заполни после осмотра ─────────────────────────────────────
BAD_COMPONENTS = {
    # 'Russian/sub1':  [0, 1],
    # 'Russian/sub2':  [2],
    # 'Spanish/sub0':  [0, 3],
}

In [ ]:
def apply_ica_and_epoch(languages=LANGUAGES):
    for language in languages:
        edf_files = sorted((BASE / language).rglob('*.edf'))
        for edf in edf_files:
            subject  = edf.parent.name
            key      = f'{language}/{subject}'
            ica_path = edf.parent / f'{subject}-ica.fif'
            out_path = edf.parent / f'{subject}-clean-epo.fif'

            if not ica_path.exists():
                print(f'[{key}] нет ICA-файла, пропускаем')
                continue

            print(f'[{key}] применяем ICA...', end=' ')
            raw = mne.io.read_raw_edf(edf, preload=True, verbose=False)
            drop = [ch for ch in EMG_CHANNELS if ch in raw.ch_names]
            raw.drop_channels(drop)
            montage = mne.channels.make_standard_montage('standard_1020')
            raw.set_montage(montage, match_case=False, on_missing='ignore')

            ica = mne.preprocessing.read_ica(ica_path)
            ica.exclude = BAD_COMPONENTS.get(key, [])
            ica.apply(raw)

            # Эпохи из аннотаций
            events, event_id = mne.events_from_annotations(raw, verbose=False)
            epochs = mne.Epochs(
                raw, events, event_id=event_id,
                tmin=TMIN, tmax=TMAX,
                baseline=BASELINE,
                preload=True, verbose=False,
            )
            epochs.save(out_path, overwrite=True)
            print(f'-> {out_path.name}  ({len(epochs)} эпох, exclude={ica.exclude})')


apply_ica_and_epoch()